# RE_302 · El 2024 va ser pitjor? · resolt

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes a l'àrea de Mobilitat de l'Ajuntament de Barcelona. La regidora ha de presentar el
balanç de sinistralitat. Tens els dos fitxers que publica la Guàrdia Urbana, un per any, amb les
mateixes columnes, i has de lliurar **una sola taula amb els dos anys** i la comparació.

**La pregunta:** comparat amb el 2023, el 2024 els accidents van ser més greus? La regidora vol el nombre de
**morts** i de **ferits greus per cada 1.000 accidents**, per a tota la ciutat i per districte.

L'enunciat complet, amb el diccionari de columnes, és a [RE_302](../RE_302_accidents_dos_anys.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Còpia congelada al repositori del curs (les dades originals són vives i canvien).
# Si tens el repositori clonat, llegeix la còpia local; si no (Colab), la baixa de GitHub.
DADES = "https://raw.githubusercontent.com/ZenidX/Machine-Learning/main/Machine%20Learning/04_reptes/dades/"

def llegeix(fitxer, **opcions):
    for ruta in ("dades/" + fitxer, "../dades/" + fitxer):
        if os.path.exists(ruta):
            return pd.read_csv(ruta, **opcions)
    return pd.read_csv(DADES + fitxer, **opcions)

a23 = llegeix("accidents_gu_bcn_2023.csv")
a24 = llegeix("accidents_gu_bcn_2024.csv")
print(a23.shape, a24.shape)
a24.head()

## 1.1 · Les mateixes columnes?

Els dos fitxers tenen les mateixes columnes amb el mateix nom. Tenen també el **mateix tipus**?
Fes una taula amb una fila per columna i el tipus de cada any al costat. Quines no coincideixen?

In [ ]:
tipus = pd.DataFrame({"2023": a23.dtypes.astype(str), "2024": a24.dtypes.astype(str)})
assert len(tipus) == 25
diferents = tipus[tipus["2023"] != tipus["2024"]]
assert set(diferents.index) == {"Numero_morts", "Numero_lesionats_lleus", "Numero_lesionats_greus",
                                "Numero_victimes", "Coordenada_UTM_Y_ED50", "Coordenada_UTM_X_ED50"}
diferents

Sis columnes canvien de tipus: les quatre de víctimes són enteres el 2023 i decimals el 2024
(un enter amb `NaN` es converteix en `float`), i les coordenades UTM són **text** el 2023.

## 1.2 · Els nuls, any per any

Fes el mateix amb els valors que falten: quants `NaN` té cada columna cada any? Per a les
columnes on la diferència és gran, mira `value_counts(dropna=False)` d'**un any i de l'altre**.
Ha canviat la realitat d'un any a l'altre, o la manera d'apuntar-la?

In [ ]:
nuls = pd.DataFrame({"2023": a23.isna().sum(), "2024": a24.isna().sum()})
nuls[(nuls["2023"] > 0) | (nuls["2024"] > 0)]

In [ ]:
for col in ["Numero_morts", "Numero_lesionats_greus", "Descripcio_causa_vianant"]:
    v23, v24 = a23[col].value_counts(dropna=False), a24[col].value_counts(dropna=False)
    assert v23.sum() == len(a23) and v24.sum() == len(a24)
    print(col, "\n  2023:", v23.head(4).to_dict(), "\n  2024:", v24.head(4).to_dict())

In [ ]:
assert (a23["Numero_morts"] == 0).sum() == 7703 and a24["Numero_morts"].isna().sum() == 7525
assert (a24["Numero_morts"] == 0).sum() == 0

**El mateix fet escrit de dues maneres.** El 2023 un accident sense morts té un `0`; el 2024 té un
`NaN` (el 2024 **no hi ha cap zero** a les columnes de víctimes). El mateix passa amb la causa del
vianant: el 2023 «No és causa del  vianant» (amb dos espais), el 2024 `NaN`. No ha canviat la
realitat, ha canviat qui omple el fitxer.

## 1.3 · Els noms dels llocs

Els districtes i els barris, s'escriuen igual els dos anys? Compara els conjunts de noms. Mira
també si els textos tenen espais de més, i si hi ha codis que no corresponguin a cap lloc.

In [ ]:
print("districtes:", set(a23["Nom_districte"]) ^ set(a24["Nom_districte"]))
print("barris (amb espais):", len(set(a23["Nom_barri"]) ^ set(a24["Nom_barri"])))
print("barris (sense espais):", set(a23["Nom_barri"].str.strip()) ^ set(a24["Nom_barri"].str.strip()))
print("codis -1 el 2023:", (a23["Codi_districte"] == -1).sum(), "| el 2024:", (a24["Codi_districte"] == -1).sum())
for nom, d in [("2023", a23), ("2024", a24)]:
    t = d.select_dtypes(exclude="number")
    sobrants = {c: int((t[c].dropna().str.strip() != t[c].dropna()).sum()) for c in t}
    print(nom, "textos amb espais sobrants:", {c: n for c, n in sobrants.items() if n})

- El 2023 hi ha un districte **`Desconegut`** (codi `-1`) amb 90 accidents que el 2024 no existeix.
- Un barri s'escriu diferent: `el Poble-sec` (2024) i `el Poble Sec` (2023). Sense arreglar-ho, el
  barri queda partit en dos.
- El 2023, el número d'expedient, el carrer i les coordenades porten **espais sobrants** (el carrer,
  omplert fins a 50 caràcters): per a pandas, `"2023S000001    "` no és `"2023S000001"`.

## 2.1 · La resposta ràpida (i desconfiada)

Ajunta les dues taules **tal com venen** i calcula, per a cada any, els morts per accident i els
ferits greus per accident (la mitjana de les columnes). Apunta els números. Te'ls creus?

In [ ]:
tot_cru = pd.concat([a23, a24], ignore_index=True)
assert len(tot_cru) == len(a23) + len(a24)
rapida = tot_cru.groupby("NK_Any")[["Numero_morts", "Numero_lesionats_greus"]].mean()
assert rapida.loc[2024, "Numero_morts"] == 1.0
rapida

Segons aquesta taula el 2024 hi ha **1 mort per accident** contra 0,003 el 2023: 370 vegades més.
És absurd. `mean()` ignora els `NaN`, i el 2024 els zeros són `NaN`: la mitjana es fa només sobre
els accidents **amb** morts, que per definició en tenen 1. Si no s'hagués mirat la 1.2, aquest
número aniria a la roda de premsa.

## 2.2 · Una funció de neteja

Escriu **una funció** `neteja(df)` que deixi qualsevol dels dos anys en un format comú: mateixos
tipus, mateixa manera d'escriure els zeros i els noms, sense espais sobrants. Fes-la servir per
als dos anys (no arreglis un any a mà). Cada decisió que prenguis, escriu-la en una línia.



- `(n23.dtypes == n24.dtypes).all()`
- cap columna de text té espais al principi o al final;
- a totes les files dels dos anys, `Numero_victimes` és la suma de lleus, greus i morts;
- la funció no esborra ni afegeix files que no hagis decidit esborrar (compta-les).

In [ ]:
VICTIMES = ["Numero_morts", "Numero_lesionats_lleus", "Numero_lesionats_greus", "Numero_victimes"]

def neteja(df):
    df = df.copy()
    df.columns = df.columns.str.strip()                              # 'Num_postal ' -> 'Num_postal'
    for col in df.select_dtypes(exclude="number").columns:          # espais sobrants i dobles
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
    df[VICTIMES] = df[VICTIMES].fillna(0).astype(int)                # el 2024 el zero és NaN
    df["Descripcio_causa_vianant"] = df["Descripcio_causa_vianant"].fillna("No és causa del vianant")
    for col in ["Coordenada_UTM_X_ED50", "Coordenada_UTM_Y_ED50"]:  # el 2023 són text
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["Nom_barri"] = df["Nom_barri"].replace({"el Poble Sec": "el Poble-sec"})
    for col in ["Codi_districte", "Codi_barri", "Codi_carrer"]:     # -1 = desconegut
        df[col] = df[col].replace(-1, np.nan).astype("Int64")
    df[["Nom_districte", "Nom_barri"]] = df[["Nom_districte", "Nom_barri"]].replace("Desconegut", np.nan)
    df["Num_postal"] = df["Num_postal"].astype("string")
    return df

n23, n24 = neteja(a23), neteja(a24)
assert (n23.dtypes == n24.dtypes).all()
for d in (n23, n24):
    t = d.select_dtypes(exclude="number")
    assert all((t[c].dropna().str.strip() == t[c].dropna()).all() for c in t)
    assert (d["Numero_victimes"] == d["Numero_lesionats_lleus"] + d["Numero_lesionats_greus"] + d["Numero_morts"]).all()
assert len(n23) == len(a23) and len(n24) == len(a24)
assert set(n23["Nom_barri"].dropna()) == set(n24["Nom_barri"].dropna())
print("tipus iguals, sense espais, víctimes quadren, cap fila perduda")

Decisions: els `NaN` de víctimes del 2024 són **0** (ho confirma que, omplerts amb 0, la suma
lleus + greus + morts dona `Numero_victimes` a totes les files dels dos anys); el `-1` i
`Desconegut` passen a `NaN` (no és un districte); el `Poble Sec` del 2023 es reescriu com el
2024. Després de netejar, els dos anys tenen exactament els mateixos barris.

## 2.3 · Un accident, una fila

El número d'expedient identifica un accident. És únic dins de cada any? I un cop netejat? Si
hi ha repetits, mira'ls sencers abans de decidir què en fas.

In [ ]:
print("expedients repetits — cru 2023:", a23["Numero_expedient"].duplicated().sum(),
      "| net 2023:", n23["Numero_expedient"].duplicated().sum(), "| 2024:", n24["Numero_expedient"].duplicated().sum())
rep = n23[n23.duplicated("Numero_expedient", keep=False)].sort_values("Numero_expedient")
print("files idèntiques senceres:", n23.duplicated().sum())
rep[["Numero_expedient", "Nom_carrer", "Hora_dia", "Numero_victimes", "Numero_vehicles_implicats"]]

In [ ]:
tot = pd.concat([n23, n24], ignore_index=True)
tot = tot.drop_duplicates()                                  # les còpies exactes (si n'hi ha)
tot = tot.drop_duplicates("Numero_expedient", keep="first")  # mateix expedient, files diferents
assert tot["Numero_expedient"].is_unique
print("files tretes:", len(n23) + len(n24) - len(tot))

El 2023 hi ha 3 expedients repetits. No són còpies exactes de la fila: és el mateix accident
amb alguna columna diferent (un mateix expedient amb dos trams de carrer, per exemple). Es queda
la primera fila de cada expedient: es perden 3 files de 15.260. És una decisió discutible
(alternativa: sumar-ne les víctimes), però l'impacte és mínim i s'ha de dir.

## 2.4 · La taula per a la regidora

Amb les dades netes, calcula per a cada any i per a cada districte: nombre d'accidents, morts i
ferits greus, i morts i ferits greus **per cada 1.000 accidents**. Decideix què fas amb els
accidents que no tenen districte conegut, i digues-ho a la taula.

In [ ]:
ciutat = tot.groupby("NK_Any").agg(accidents=("Numero_expedient", "size"),
                                    morts=("Numero_morts", "sum"),
                                    greus=("Numero_lesionats_greus", "sum"))
ciutat["morts_1000"] = (1000 * ciutat["morts"] / ciutat["accidents"]).round(2)
ciutat["greus_1000"] = (1000 * ciutat["greus"] / ciutat["accidents"]).round(1)
# Propietats: els accidents són les files; netejar no crea ni fa desaparèixer morts
assert ciutat["accidents"].sum() == len(tot)
assert ciutat.loc[2023, "morts"] == a23["Numero_morts"].sum() == 21
assert ciutat.loc[2024, "morts"] == a24["Numero_morts"].sum() == 11
ciutat

In [ ]:
dist = (tot.assign(Nom_districte=tot["Nom_districte"].fillna("(sense districte)"))
          .groupby(["Nom_districte", "NK_Any"])
          .agg(accidents=("Numero_expedient", "size"), morts=("Numero_morts", "sum"),
               greus=("Numero_lesionats_greus", "sum")))
dist["greus_1000"] = (1000 * dist["greus"] / dist["accidents"]).round(1)
dist["morts_1000"] = (1000 * dist["morts"] / dist["accidents"]).round(2)
assert dist["accidents"].sum() == len(tot)
taula_d = dist[["accidents", "morts", "greus_1000"]].unstack("NK_Any")
taula_d["dif_greus_1000"] = taula_d[("greus_1000", 2024)] - taula_d[("greus_1000", 2023)]
taula_d.sort_values("dif_greus_1000")

## 2.5 · La resposta

Respon la regidora: el 2024 els accidents van ser més greus que el 2023? Quins districtes han
canviat més? Compara-ho amb el que havies dit a la tasca 2.1.

In [ ]:
r = ciutat
print(f"Morts: {r.loc[2023,'morts']} -> {r.loc[2024,'morts']}  ·  per 1.000 accidents: {r.loc[2023,'morts_1000']} -> {r.loc[2024,'morts_1000']}")
print(f"Ferits greus: {r.loc[2023,'greus']} -> {r.loc[2024,'greus']}  ·  per 1.000: {r.loc[2023,'greus_1000']} -> {r.loc[2024,'greus_1000']}")
assert r.loc[2024, "morts_1000"] < r.loc[2023, "morts_1000"]
assert r.loc[2024, "greus_1000"] > r.loc[2023, "greus_1000"]

**Resposta:** el 2024 hi va haver **menys morts** (11 contra 21; 1,46 per 1.000 accidents contra
2,72) però **més ferits greus per accident** (254 contra 226; 33,7 per 1.000 contra 29,3). No es pot dir
«més greus» ni «menys greus» sense separar les dues coses. Per districtes els canvis són de poques
desenes de casos i amb morts d'un dígit: cal presentar-los amb els números absoluts al costat.

La resposta ràpida de la 2.1 (1 mort per accident el 2024) era **falsa per un problema de format**,
no de dades. Les 90 files del 2023 sense districte es mostren a part.

## 3.1 · On són

Dibuixa els accidents amb la longitud i la latitud, un color per any. Les columnes de
coordenades UTM, diuen el mateix els dos anys? Quina faries servir?

In [ ]:
fig, eix = plt.subplots(1, 2, figsize=(11, 5))
for any_, d in tot.groupby("NK_Any"):
    eix[0].scatter(d["Longitud_WGS84"], d["Latitud_WGS84"], s=1, alpha=0.3, label=any_)
eix[0].legend(); eix[0].set_title("Accidents per any")
eix[1].scatter(tot["Coordenada_UTM_X_ED50"], tot["Coordenada_UTM_Y_ED50"], s=1, alpha=0.3)
eix[1].set_xlabel("columna 'X'"); eix[1].set_ylabel("columna 'Y'"); eix[1].set_title("Les columnes UTM tal com es diuen")
print(tot[["Coordenada_UTM_X_ED50", "Coordenada_UTM_Y_ED50"]].describe().loc[["min", "max"]])

Les columnes UTM estan **intercanviades** els dos anys: la «X» té valors de 4,58 milions, que és
una coordenada nord (Y). Per dibuixar, la longitud i la latitud WGS84 són coherents i n'hi ha prou.